In [1]:
import deeplabcut
import glob
import os
import numpy as np


Loading DLC 2.3.9...


## Paths to DLC configs on 'The Hook'
```
r"D:\DLC_analysis\30cm_open_field-berkowitz-2022-07-14\config.yaml"
r"D:\DLC_analysis\40cm_open_field-berkowitz-2022-07-22\config.yaml"
r"D:\DLC_analysis\40cm_OR-berkowitz-2023-02-17\config.yaml"
r"D:\DLC_analysis\aya_poorlight-berkowitz-2024-01-08\config.yaml"
r"D:\DLC_analysis\dim_cpp-berkowitz-2023-10-10\config.yaml"
r"D:\DLC_analysis\linear_track-Berkowitz-2021-11-15\config.yaml"
r"D:\DLC_analysis\mouse_close_camera_multi_maze-ryan-2023-01-03\config.yaml"
r"D:\DLC_analysis\mouse_cpp-berkowitz-2023-06-28\config.yaml"
r"D:\DLC_analysis\multianimal_OL-berkowitz-2022-04-30\config.yaml"

```

## Set path to data directory and use glob to get paths to all video files 

In [2]:
import ruamel.yaml

def update_config_cropping(config_path, xmin, xmax, ymin, ymax):

    yaml = ruamel.yaml.YAML()
    yaml.preserve_quotes = True  # keeps formatting stable if desired

    # ---- Read existing config ----
    with open(config_path, "r") as fp:
        data = yaml.load(fp)

    if data is None:
        raise ValueError("YAML file is empty or malformed.")

    # ---- Early exit if already set ----
    if (
        data.get("x1") == xmin and
        data.get("x2") == xmax and
        data.get("y1") == ymin and
        data.get("y2") == ymax and
        data.get("cropping") is True
    ):
        print(f"Cropping parameters already set to ({xmin}, {xmax}, {ymin}, {ymax}). No update needed.")
        return data
    
    # ---- Update cropping parameters ----
    data["cropping"] = True
    data["x1"] = int(xmin)
    data["x2"] = int(xmax)
    data["y1"] = int(ymin)
    data["y2"] = int(ymax)

    # ---- Write safely ----
    with open(config_path, "w") as f:
        yaml.dump(data, f)
        f.flush()
        os.fsync(f.fileno())  # ensure write completes to disk

In [4]:
# load check_dlc.csv 
import pandas as pd
basepaths = pd.read_csv(r"Y:\laura_berkowitz\behavior_validation\appps1_cpp\analysis\keypoint_moseq\check_dlc.csv")
basepaths = basepaths.query("~basepath.str.contains('pair')").copy().reset_index(drop=True)
basepaths

,basepath,y_min,y_max,x_min,x_max,config_path,checked_status
0,Y:\laura_berkowitz\behavior_validation\appps1_...,170.118305,879.448931,96.996171,1221.329363,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,done
1,Y:\laura_berkowitz\behavior_validation\appps1_...,170.118305,879.448931,96.996171,1221.329363,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,NaN
2,Y:\laura_berkowitz\behavior_validation\appps1_...,3.098971,721.477857,166.954868,1378.770633,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,NaN
3,Y:\laura_berkowitz\behavior_validation\appps1_...,170.118305,879.448931,96.996171,1221.329363,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,NaN
4,Y:\laura_berkowitz\behavior_validation\appps1_...,2.351310,744.058023,220.613957,1392.039638,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,NaN
5,Y:\laura_berkowitz\behavior_validation\appps1_...,2.351310,744.058023,220.613957,1392.039638,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,NaN
6,Y:\laura_berkowitz\behavior_validation\appps1_...,2.351310,744.058023,220.613957,1392.039638,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,NaN
7,Y:\laura_berkowitz\behavior_validation\appps1_...,3.098971,721.477857,166.954868,1378.770633,Y:\laura_berkowitz\dlc_models\ephys1_cpp2024-b...,NaN
8,Y:\laura_berkowitz\behavior_validation\appps1_...,3.098971,721.477857,166.954868,1378.770633,Y:\laura_berkowitz\dlc_models\ephys1_cpp2024-b...,NaN
9,Y:\laura_berkowitz\behavior_validation\appps1_...,3.098971,721.477857,166.954868,1378.770633,Y:\laura_berkowitz\dlc_models\ephys1_cpp2024-b...,NaN


In [6]:

# Data path 
data_path = r'Y:\laura_berkowitz\behavior_validation\appps1_cpp\data\cohort5\3891\3891_cpptask_day03'

# use glob to find path to all .avi files 
video_files = glob.glob(data_path + '/**/*.avi', recursive=True)
video_files = [os.path.normpath(v) for v in video_files if 'pair' not in v]
video_files

['Y:\\laura_berkowitz\\behavior_validation\\appps1_cpp\\data\\cohort5\\3891\\3891_cpptask_day03\\3891_cppbaseline_day03-12052023090906.avi',
 'Y:\\laura_berkowitz\\behavior_validation\\appps1_cpp\\data\\cohort5\\3891\\3891_cpptask_day03\\3891_cpptest_day03-12052023104912.avi']

## Set DLC config and analyze videos. If video is already run (H5 or CSV in folder, DLC will continue to next video)

In [4]:
basepaths

,basepath,y_min,y_max,x_min,x_max,config_path,checked_status
0,Y:\laura_berkowitz\behavior_validation\appps1_...,170.118305,879.448931,96.996171,1221.329363,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,done
1,Y:\laura_berkowitz\behavior_validation\appps1_...,170.118305,879.448931,96.996171,1221.329363,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,done
2,Y:\laura_berkowitz\behavior_validation\appps1_...,3.098971,721.477857,166.954868,1378.770633,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,done
3,Y:\laura_berkowitz\behavior_validation\appps1_...,170.118305,879.448931,96.996171,1221.329363,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,done
4,Y:\laura_berkowitz\behavior_validation\appps1_...,2.351310,744.058023,220.613957,1392.039638,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,done
5,Y:\laura_berkowitz\behavior_validation\appps1_...,2.351310,744.058023,220.613957,1392.039638,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,done
6,Y:\laura_berkowitz\behavior_validation\appps1_...,2.351310,744.058023,220.613957,1392.039638,Y:\laura_berkowitz\dlc_models\dim_cpp-berkowit...,done
7,Y:\laura_berkowitz\behavior_validation\appps1_...,3.098971,721.477857,166.954868,1378.770633,Y:\laura_berkowitz\dlc_models\ephys1_cpp2024-b...,done
8,Y:\laura_berkowitz\behavior_validation\appps1_...,3.098971,721.477857,166.954868,1378.770633,Y:\laura_berkowitz\dlc_models\ephys1_cpp2024-b...,done
9,Y:\laura_berkowitz\behavior_validation\appps1_...,3.098971,721.477857,166.954868,1378.770633,Y:\laura_berkowitz\dlc_models\ephys1_cpp2024-b...,done


In [ ]:
# load check_dlc.csv
import pandas as pd

basepaths = pd.read_csv(
    r"Y:\laura_berkowitz\behavior_validation\appps1_cpp\analysis\keypoint_moseq\check_dlc.csv"
)

# loop in unique config_path and 
for i, basepath in enumerate(basepaths["basepath"].values):
    if 'done' == basepaths.loc[i, "checked_status"]:
        continue  # skip if this basepath has already been processed

    config_path = os.path.normpath(basepaths.loc[i, "config_path"])
    xmin = int(basepaths.loc[i, "x_min"])
    xmax = int(basepaths.loc[i, "x_max"])
    ymin = int(basepaths.loc[i, "y_min"])
    ymax = int(basepaths.loc[i, "y_max"])

    update_config_cropping(config_path, xmin, xmax, ymin, ymax)

    # use glob to find path to all .avi files
    video_files = glob.glob(basepath + "/**/*.avi", recursive=True)
    video_files = [os.path.normpath(v) for v in video_files if 'pair' not in v]  # filter out any videos with 'pair' in the name
    deeplabcut.analyze_videos(
        config_path,
        video_files,
        videotype="avi",
        shuffle=1,
        trainingsetindex=0,
        save_as_csv=True,
        dynamic=(False, 0.8, 100),
    )

    # deeplabcut.filterpredictions(config_path, video_files)
    deeplabcut.filterpredictions(config_path, video_files)

    basepaths.loc[i, "checked_status"] = "done"

In [ ]:
#load the dataframe 
import pandas as pd 
df = pd.read_csv(r"Y:\laura_berkowitz\app_ps1_ephys\DLC_status_sessions.csv")
df
# use os.path.normpath to fix filesep differences within df.basepath 
df.basepath=df.basepath.apply(lambda x: os.path.normpath(x))
df.basepath

In [34]:

#iterate through all the DLC models
dlc_models= df['dlc_model'].unique()
print(dlc_models)

['none' '30cm_open_field-berkowitz-2022-07-14'
 'ephys1_dim-berkowitz-2024-08-22' 'linear_track-Berkowitz-2021-11-15'
 'ephys1_linear-berkowitz-2024-08-19'
 '40cm_open_field-berkowitz-2022-07-22']


In [ ]:
base_config_dir= r"Y:\laura_berkowitz\dlc_models"


for model in dlc_models:

    if (model=='none') or ((model) == float):
        continue 
    print('running videos for model:', model)

    temp_df=df[(df['dlc_model']== model) & (df['DLC_flag']==False)]
    config_path = os.path.join(base_config_dir, model, 'config.yaml')

    #in temp df need to join basepath and filename and add .avi to filename 
    temp_df['video_path']= temp_df['basepath'] + '\\' + temp_df['file_name'] + '.avi'

    # make a list of video file paths from video_path column
    # video_files = list
    video_files= temp_df['video_path'].tolist()

    deeplabcut.analyze_videos(config_path, video_files, videotype='mpg', shuffle=1, trainingsetindex=0, save_as_csv=True)
    deeplabcut.filterpredictions(config_path, video_files)

    # update the dlc flag to true
    df.loc[(df['dlc_model']== model) & (df['DLC_flag']==False),'DLC_flag']=True

    df.to_csv(r"Y:\laura_berkowitz\app_ps1_ephys\DLC_status_sessions.csv", index=False)
# save the updated df 
df.to_csv(r"Y:\laura_berkowitz\app_ps1_ephys\DLC_status_sessions.csv", index=False)